In [ ]:
# reading the data
import pandas as pd

# Load dataset
df = pd.read_csv("twcs.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (2811774, 7)
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3 

In [2]:
df["author_id"] = df["author_id"].astype(str)

In [3]:
# Keep tweets from AmazonHelp and customer tweets that mention AmazonHelp

amazon_df = df[
    (df["author_id"] == "AmazonHelp") |
    (df["text"].str.contains("@AmazonHelp", case=False, na=False))
].copy()

print("Amazon-related tweets:", len(amazon_df))

Amazon-related tweets: 305008


In [4]:
# %%
amazon_df["clean_text"] = (
    amazon_df["text"]
    .str.replace(r"http\S+|www\S+", "", regex=True)
    .str.replace(r"@\w+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [5]:
# %%
customer_df = amazon_df[
    amazon_df["inbound"] == True
].copy()

amazon_response_df = amazon_df[
    (amazon_df["author_id"] == "AmazonHelp") &
    (amazon_df["inbound"] == False)
].copy()

print("Customer tweets:", len(customer_df))
print("Amazon responses:", len(amazon_response_df))

Customer tweets: 135160
Amazon responses: 169840


In [6]:
# %%
customer_df = customer_df.rename(columns={
    "tweet_id": "customer_tweet_id",
    "author_id": "customer_id",
    "text": "customer_query",
    "clean_text": "clean_customer_query"
})

amazon_response_df = amazon_response_df.rename(columns={
    "tweet_id": "amazon_tweet_id",
    "in_response_to_tweet_id": "customer_tweet_id",
    "text": "amazon_response",
    "clean_text": "clean_amazon_response"
})

In [7]:
# %%
conversation_df = amazon_response_df.merge(
    customer_df[
        [
            "customer_tweet_id",
            "customer_id",
            "customer_query",
            "clean_customer_query"
        ]
    ],
    on="customer_tweet_id",
    how="inner"
)

print("Matched customer → Amazon pairs:", len(conversation_df))

Matched customer → Amazon pairs: 100015


In [8]:
# %%
intent_df = conversation_df[
    [
        "customer_tweet_id",
        "customer_id",
        "customer_query",
        "clean_customer_query",
        "amazon_tweet_id",
        "amazon_response"
    ]
].copy()

print("Initial intent dataset:", len(intent_df))

Initial intent dataset: 100015


In [9]:
# %%
non_informative = {
    "yes",
    "yes.",
    "no",
    "no.",
    "done",
    "done.",
    "thanks",
    "thanks!",
    "thanks.",
    "thank you",
    "thank you!",
    "thank you.",
    "merci",
    "gracias",
    "oui",
    "ok",
    "ok.",
    "okay",
    "okay.",
    "amazon",
    ".com"
}

intent_df["query_check"] = (
    intent_df["clean_customer_query"]
    .str.lower()
    .str.strip()
)

intent_df = intent_df[
    ~intent_df["query_check"].isin(non_informative)
].copy()

intent_df.drop(columns=["query_check"], inplace=True)

print("Final intent dataset:", len(intent_df))
print(
    "Unique queries:",
    intent_df["clean_customer_query"].nunique()
)

Final intent dataset: 99184
Unique queries: 89715


In [10]:
# %%
print(intent_df.head(10).to_string(index=False))

 customer_tweet_id customer_id                                                                                                                                                     customer_query                                                                                                                    clean_customer_query  amazon_tweet_id                                                                                                                      amazon_response
             271.0      115770                                                                                                   @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。                                                                                     電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんでしょうね。              273                                                                @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
             274.0      115770          

In [11]:
# %%
print(
    intent_df["clean_customer_query"]
    .sample(100, random_state=42)
    .to_string(index=False)
)

       Sunday 4:48 AM Arrival Scan WINDSOR, ON, CA
J'ai bien compris et je viens de le faire une 2...
Does sorry make sense? Get me the damages sir. ...
I don't want to trigger this prematurely, but i...
Dernière mise à jour : mercredi 25 octobre 11:1...
Hi Amazon, thank you for your reply we are now ...
Please help to get my order , had placed it 10 ...
                         😂 faremo un altro ordine💪
Hw to speak to Custmr care exu I need to place ...
is there any reason you've suddenly started to ...
Do you really want me to be describing your poo...
This is utter rubbish. How can you have the sam...
30 de Noviembre. Pero se que ustedes son los me...
Not sure since when I check sometimes it tells ...
A game that I have pre ordered is appearing as ...
since yesterday i have called 20 times at your ...
I have Amazon Prime but I can't figure out how ...
Bonjour Après avoir contacté le SAV, j'ai pris ...
I don't see exchange option for One Plus either...
                     Yep. It ju

In [12]:
# %%
intent_df.to_csv(
    "amazon_intent_base.csv",
    index=False
)

print(
    f"Saved {len(intent_df)} rows to amazon_intent_base.csv"
)

Saved 99184 rows to amazon_intent_base.csv


In [13]:
print(intent_df.head().to_string())

   customer_tweet_id customer_id                                                                                                                                        customer_query                                                                                                                     clean_customer_query  amazon_tweet_id                                                                                                                  amazon_response
0              271.0      115770                                                                                      @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。                                                                                      電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんでしょうね。              273                                                            @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
1              274.0      115770                                    